# Fan-out latency — example analysis

One run of the harness: 200k messages at 250k msg/s, mixed types, 1024 slots.

```bash
make -C ../harness
../harness/bin/producer --shm /fanout_demo --slots 1024 --count 0 --rate 250000 --type mixed &
../harness/bin/consumer --shm /fanout_demo --slots 1024 --from-edge \
    --count 200000 --csv data/latency.csv
```

In [7]:
import numpy as np
import pandas as pd
from bokeh.io import output_notebook, show
from bokeh.models import ColumnDataSource, HoverTool, NumeralTickFormatter
from bokeh.plotting import figure

output_notebook()

BLUE = "#2a78d6"

ns = pd.read_csv("./harness/scripts/latencies.csv")["latency_ns"].to_numpy()
ns.size, ns.min(), int(ns.mean()), ns.max()

Loading BokehJS ...

(4992351, np.int64(76), 1682, np.int64(5117035))

## Distribution

Clipped at p99.9 for display purposes

In [8]:
clip = np.percentile(ns, 99.9)
counts, edges = np.histogram(ns[ns <= clip], bins=60)
src = ColumnDataSource(dict(count=counts, left=edges[:-1], right=edges[1:],
                            mid=(edges[:-1] + edges[1:]) / 2))

p = figure(height=320, sizing_mode="stretch_width",
           title=f"Delivery latency, clipped at p99.9 = {clip:,.0f} ns",
           x_axis_label="latency (ns)", y_axis_label="messages",
           tools="pan,box_zoom,reset,save")
p.quad(source=src, bottom=0, top="count", left="left", right="right",
       fill_color=BLUE, line_color="white")
p.y_range.start = 0
p.xaxis.formatter = p.yaxis.formatter = NumeralTickFormatter(format="0,0")
p.add_tools(HoverTool(tooltips=[("latency", "@mid{0,0} ns"),
                                ("messages", "@count{0,0}")], mode="vline"))
show(p)

## Tail

Log axis on both scales — a mean hides the tail entirely.

In [9]:
# Plotted against "distance from 100th percentile" on a log axis, so the tail gets
# real width instead of being squeezed into the last few pixels.
# Stops at p99.99: beyond that is a handful of samples out of 200k, not a stable figure.
pcts = np.array([50, 75, 90, 99, 99.9, 99.99])
curve = ColumnDataSource(dict(x=100 - pcts, ns=np.percentile(ns, pcts),
                              pct=[f"p{v:g}" for v in pcts]))

q = figure(height=320, sizing_mode="stretch_width",
           x_axis_type="log", y_axis_type="log",
           x_range=(70, 0.0025),  # reversed: p50 on the left, p99.99 on the right
           title="Latency tail",
           x_axis_label="percentile", y_axis_label="latency (ns, log)",
           tools="pan,box_zoom,reset,save")
q.line(source=curve, x="x", y="ns", color=BLUE, width=2)
q.scatter(source=curve, x="x", y="ns", color=BLUE, size=9,
          line_color="white", line_width=2)
q.xaxis.ticker = list(100 - pcts)
q.xaxis.major_label_overrides = {100 - v: f"p{v:g}" for v in pcts}
q.yaxis.ticker = [100, 200, 500, 1000, 2000, 5000]
q.yaxis.formatter = NumeralTickFormatter(format="0,0")
q.add_tools(HoverTool(tooltips=[("percentile", "@pct"), ("latency", "@ns{0,0} ns")]))
show(q)

pd.DataFrame([{f"p{v:g}": int(x) for v, x in zip(pcts, np.percentile(ns, pcts))}])

,p50,p75,p90,p99,p99.9,p99.99
0,208,253,314,391,4293,4002183


## Latency over the run

The CSV only has `seq,latency_ns` — no wall-clock receive time — so the x-axis
below is **arrival order** (message #), used as a proxy for elapsed time. It's
exact for a constant-rate producer; under `--rate 0` (unthrottled) it's still
monotonic in time but not evenly spaced in seconds.

Raw per-message scatter is unreadable/too slow at millions of points, so the
run is split into ~400 equal-count buckets and each bucket's p50/p99/p99.9 is
plotted as a percentile envelope.

In [10]:
from bokeh.models import Label

n_bins = min(1000, ns.size)
edges = np.linspace(0, ns.size, n_bins + 1).astype(int)
mid = (edges[:-1] + edges[1:]) / 2

p50 = np.empty(n_bins)
p99 = np.empty(n_bins)
p999 = np.empty(n_bins)
for i in range(n_bins):
    seg = ns[edges[i]:edges[i + 1]]
    p50[i], p99[i], p999[i] = np.percentile(seg, [50, 99, 99.9])

src = ColumnDataSource(dict(x=mid, p50=p50, p99=p99, p999=p999))

t = figure(height=340, sizing_mode="stretch_width",
           title="Latency over the run (binned percentile envelope)",
           x_axis_label="message # (arrival order, proxy for elapsed time)",
           y_axis_label="latency (ns)",
           tools="pan,box_zoom,reset,save")
t.varea(source=src, x="x", y1=0, y2="p999", fill_color=BLUE, fill_alpha=0.15)
t.varea(source=src, x="x", y1=0, y2="p99", fill_color=BLUE, fill_alpha=0.25)
t.line(source=src, x="x", y="p50", color=BLUE, line_width=2)
t.y_range.start = 0
t.xaxis.formatter = t.yaxis.formatter = NumeralTickFormatter(format="0,0")
t.add_tools(HoverTool(tooltips=[("msg #", "@x{0,0}"), ("p50", "@p50{0,0} ns"),
                                ("p99", "@p99{0,0} ns"), ("p99.9", "@p999{0,0} ns")],
                      mode="vline"))

for name, y in [("p50", p50[-1]), ("p99", p99[-1]), ("p99.9", p999[-1])]:
    t.add_layout(Label(x=mid[-1], y=y, text=f" {name}", text_color=BLUE,
                       text_font_size="10px", x_offset=4, text_baseline="middle"))

show(t)

In [11]:
window = max(10, ns.size // 200_000)  # cap plotted points around 200k
n = (ns.size // window) * window
agg = ns[:n].reshape(-1, window).mean(axis=1)
x = np.arange(agg.size) * window + window / 2

src = ColumnDataSource(dict(x=x, y=agg))

w = figure(height=340, sizing_mode="stretch_width",
           title=f"Latency over the run (raw mean, window={window} msgs)",
           x_axis_label="message # (arrival order, proxy for elapsed time)",
           y_axis_label="latency (ns)",
           tools="pan,box_zoom,reset,save")
w.line(source=src, x="x", y="y", color=BLUE, line_width=1.5)
w.y_range.start = 0
w.xaxis.formatter = w.yaxis.formatter = NumeralTickFormatter(format="0,0")
w.add_tools(HoverTool(tooltips=[("msg #", "@x{0,0}"), ("mean latency", "@y{0,0} ns")],
                      mode="vline"))
show(w)